# cellmap-flow on Colab (production dashboard, nginx single-port fan-out)

Runs the production cellmap-flow dashboard, Neuroglancer, and inference
server in this Colab session, all multiplexed behind one Colab proxyPort
URL via nginx.

```
                       Colab container (no public IP)
                       ┌──────────────────────────────────────────────┐
                       │  nginx          on :8501  ←── proxyPort      │
                       │     ├─ /                  → dashboard :5000  │
                       │     ├─ /ng/<…>            → NG tornado :9090 │
                       │     └─ /infer/<…>         → inference :8765  │
                       └──────────────────────────────────────────────┘
```

Why one port: the dashboard iframes Neuroglancer. If NG were on its
own proxyPort URL, it'd be a different subdomain → 3rd-party iframe →
Chrome blocks the Colab auth cookie → NG fails to load. By putting
everything behind one nginx → one proxyPort URL, the iframe is
same-origin and the cookie flows naturally.

This also kills the ~150 ms cloudflared tunnel hop that older versions
of this notebook added to every chunk request.

**Limit**: the proxyPort URL only works for **your own Colab session**.
For a shareable demo, see the HF Docker Space path (separate doc).

**Steps:**
1. Runtime → Change runtime type → **T4 GPU**.
2. **Run all cells.** If the install cell restarts the kernel, click Run All again.
3. Click the printed Dashboard URL — that's the only URL you need.

## 1. Install

In [ ]:
# Skip-if-already-installed: after the (inevitable) first-run kernel
# restart, the second pass of Run-All sees cellmap_flow already
# present and skips pip entirely (~5s instead of ~90s).
def _all_packages_installed():
    try:
        import cellmap_flow.globals
        import bioimageio.core, bioimageio.spec
        # Must match what we force-reinstall below.
        return (
            bioimageio.core.__version__ == "0.9.6"
            and bioimageio.spec.__version__.startswith("0.5.7")
        )
    except Exception:
        return False

import os, subprocess

if _all_packages_installed():
    print("[install] python packages already at required versions — skipping pip.")
else:
    print("[install] running pip (this is the slow first-run step) ...")
    # Let pip pick a consistent numpy/scipy/skimage set — the kernel
    # auto-restart below handles any numpy ABI change vs. Colab's
    # pre-imported numpy.
    %pip install -q "cellmap-flow[bioimageio] @ git+https://github.com/janelia-cellmap/cellmap-flow.git@browser-inference" huggingface_hub s3fs
    %pip install -q --force-reinstall "bioimageio.core==0.9.6" "bioimageio.spec==0.5.7.4"

# nginx — single-port fan-out (see top-level docstring for why).
if not os.path.exists("/usr/sbin/nginx"):
    print("[install] apt-installing nginx ...")
    subprocess.check_call(
        ["apt-get", "install", "-y", "-q", "nginx"],
        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )
    print("[install] nginx installed.")
else:
    print("[install] nginx already present.")

# Pre-download the default BMZ model so the first inference doesn't
# add another minute on top of pip. Cached after first run.
try:
    print("[install] pre-fetching bioimageio model 'hiding-blowfish' (cached on re-run) ...")
    import bioimageio.spec
    bioimageio.spec.load_description("hiding-blowfish")
    print("[install] model pre-fetched.")
except Exception as e:
    print(f"[install] could not pre-fetch model (will lazy-load on first inference): {e}")

# Pip can mutate numpy on disk, while Colab pre-imported the old
# numpy at kernel startup. If those don't match, the next import
# chain will ImportError. Detect + force-restart; user clicks Run
# All again. On the second pass, the skip-if-already-installed
# check above means pip is a no-op so this branch won't trigger.
try:
    import cellmap_flow.globals
    print("[install] kernel is in sync with on-disk packages, ready to proceed.")
except Exception as e:
    print(f"[install] kernel state mismatched on-disk packages ({e.__class__.__name__}: {e}).")
    print("[install] restarting kernel in 3s — when it comes back, click Run All again.")
    import time
    time.sleep(3)
    os.kill(os.getpid(), 9)

## 2. Configure model + dataset

`MODEL_TYPE = "huggingface"` for a cellmap HF model, or `"bioimage"`
for a BMZ model.

T4 sizing notes:
- 178³ HF models (`fly_organelles_run07_*`): fit cleanly.
- 288³ HF models (`jrc_mus-livers_*`): borderline, OOM-prone.
- 2D BMZ models (`hiding-blowfish`): trivial, fits anywhere.


In [ ]:
MODEL_TYPE = "bioimage"      # or "huggingface"

# --- Mode A: huggingface ---
HF_REPO = "cellmap/fly_organelles_run07_432000"
HF_NAME = HF_REPO.split("/")[-1]
HF_DATASET = (
    "https://janelia-cosem-datasets.s3.amazonaws.com/"
    "jrc_mus-liver/jrc_mus-liver.zarr/recon-1/em/fibsem-uint8"
)

# --- Mode B: bioimage (BMZ) ---
BMZ_MODEL = "hiding-blowfish"
BMZ_VOXEL_SIZE = "8,8,8"
BMZ_DATASET = (
    "https://janelia-cosem-datasets.s3.amazonaws.com/"
    "jrc_hela-2/jrc_hela-2.zarr/recon-1/em/fibsem-uint8/s1"
)

# All ports are *internal* to the Colab container except PUBLIC_PORT,
# which is the single port that gets exposed via Colab's proxyPort.
PUBLIC_PORT    = 8501   # nginx — the one that proxyPort sees
DASHBOARD_PORT = 5000   # internal: dashboard Flask
NG_PORT        = 9090   # internal: NG-Python tornado
INFERENCE_PORT = 8765   # internal: cellmap_flow_server Flask

if MODEL_TYPE == "huggingface":
    MODEL_NAME = HF_NAME
    DATASET = HF_DATASET
elif MODEL_TYPE == "bioimage":
    MODEL_NAME = BMZ_MODEL
    DATASET = BMZ_DATASET
else:
    raise ValueError(f"unknown MODEL_TYPE={MODEL_TYPE!r}")

print(f"MODEL_TYPE = {MODEL_TYPE}")
print(f"MODEL_NAME = {MODEL_NAME}")
print(f"DATASET    = {DATASET}")

## 3. Start the cellmap-flow inference server (subprocess)

In [ ]:
import os, subprocess, time, requests

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

if MODEL_TYPE == "huggingface":
    cmd = [
        "cellmap_flow_server", "huggingface",
        "--repo", HF_REPO, "--name", HF_NAME,
        "-d", HF_DATASET, "--port", str(INFERENCE_PORT),
    ]
elif MODEL_TYPE == "bioimage":
    cmd = [
        "cellmap_flow_server", "bioimage",
        "--model-name", BMZ_MODEL, "--voxel-size", BMZ_VOXEL_SIZE,
        "--name", BMZ_MODEL,
        "-d", BMZ_DATASET, "--port", str(INFERENCE_PORT),
    ]
print("starting:", " ".join(cmd))
server = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    env={**os.environ},
)
print(f"server pid={server.pid}, waiting for it to listen on :{INFERENCE_PORT} ...")
for _ in range(300):
    line = server.stdout.readline()
    if not line: time.sleep(0.25); continue
    print(line, end="")
    if "Running on" in line or f":{INFERENCE_PORT}" in line:
        print("\n[server] ready.")
        break

# Warm-up: hit a chunk URL once so the GPU loads the model and runs
# its first forward pass NOW (while later cells set up nginx + NG)
# rather than when the user first clicks the dashboard URL.
print("[warm-up] triggering a chunk to load model + warm GPU ...")
t0 = time.time()
try:
    r = requests.get(
        f"http://localhost:{INFERENCE_PORT}/{MODEL_NAME}/s0/0.0.0.0",
        timeout=180,
    )
    print(f"[warm-up] done in {time.time()-t0:.1f}s (status {r.status_code}, {len(r.content)} bytes)")
except Exception as e:
    print(f"[warm-up] failed (not fatal — will warm on first real request): {e}")

## 4. Start nginx as the single-port front door

Writes an nginx config that path-routes one external port to three
internal services, then starts nginx. After this, dashboard, NG, and
inference are all reachable through one Colab proxyPort URL.

In [ ]:
nginx_conf = f"""
worker_processes 1;
events {{}}
http {{
    # Generous defaults — chunk uploads + state sync can be slow.
    client_max_body_size 256m;
    proxy_read_timeout 86400;
    proxy_send_timeout 86400;
    proxy_buffering off;

    server {{
        listen {PUBLIC_PORT};

        # Neuroglancer tornado server — path prefix /ng/. The trailing
        # slash on proxy_pass strips /ng/ before forwarding, so the
        # tornado side sees its native paths (/v/<token>/...).
        location /ng/ {{
            proxy_pass http://127.0.0.1:{NG_PORT}/;
            proxy_http_version 1.1;
            proxy_set_header Host $host;
            proxy_set_header Upgrade $http_upgrade;
            proxy_set_header Connection "upgrade";  # for NG state-sync websocket
        }}

        # Inference Flask — path prefix /infer/. Strip /infer/.
        location /infer/ {{
            proxy_pass http://127.0.0.1:{INFERENCE_PORT}/;
            proxy_http_version 1.1;
            proxy_set_header Host $host;
        }}

        # Dashboard Flask — root. NB no trailing slash on proxy_pass
        # so paths are preserved (e.g. /api/process stays /api/process).
        location / {{
            proxy_pass http://127.0.0.1:{DASHBOARD_PORT};
            proxy_http_version 1.1;
            proxy_set_header Host $host;
            proxy_set_header Upgrade $http_upgrade;
            proxy_set_header Connection "upgrade";  # dashboard SSE
        }}
    }}
}}
"""
import subprocess, os, tempfile, time
conf_path = "/tmp/cellmap_flow_nginx.conf"
with open(conf_path, "w") as fh:
    fh.write(nginx_conf)

# Stop any nginx left over from a previous Run All, then start fresh.
subprocess.run(["nginx", "-s", "quit", "-c", conf_path],
                stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
time.sleep(0.5)
# Use a writable pid file inside /tmp (Colab can't write /run/nginx.pid as non-root).
subprocess.check_call([
    "nginx",
    "-c", conf_path,
    "-g", "pid /tmp/cellmap_flow_nginx.pid; daemon on;",
])
print(f"[nginx] listening on :{PUBLIC_PORT}")
print(f"        /          → dashboard :{DASHBOARD_PORT}")
print(f"        /ng/<...>  → NG tornado :{NG_PORT}")
print(f"        /infer/... → inference :{INFERENCE_PORT}")

## 5. Configure Neuroglancer + start the dashboard

Binds NG-Python to an internal port, monkey-patches its URL emission so
str(viewer) returns the public-facing /ng/<path> URL (so the dashboard
iframe loads NG from the same origin as itself), then runs the dashboard
Flask app in a background thread.

In [ ]:
import neuroglancer, threading, urllib.parse, re, requests
import neuroglancer.server as _ng_server_mod
from cellmap_flow.globals import g
from cellmap_flow.dashboard.app import app
from cellmap_flow.dashboard import state
from cellmap_flow.image_data_interface import ImageDataInterface
from cellmap_flow.utils.scale_pyramid import ScalePyramid

# Monkey-patch get_raw_layer for remote (http/s3/gs) zarr URLs:
# upstream uses os.listdir() to enumerate /sN scale dirs which doesn't
# work over HTTP. Read OME-NGFF /.zattrs instead and build the same
# prod-shape output (ScalePyramid of LocalVolumes via
# ImageDataInterface, so input normalizers still apply to the raw).
import cellmap_flow.utils.scale_pyramid as _sp
import cellmap_flow.dashboard.routes.pipeline as _pipe
import cellmap_flow.utils.neuroglancer_utils as _ngu
_orig_get_raw_layer = _sp.get_raw_layer

def _remote_safe_get_raw_layer(dataset_path, normalize=True, wrap_raw=True):
    if not dataset_path.startswith(("http://", "https://", "s3://", "gs://")):
        return _orig_get_raw_layer(dataset_path, normalize=normalize, wrap_raw=wrap_raw)
    group_url = re.sub(r"/s\d+/?$", "", dataset_path)
    if not wrap_raw:
        return neuroglancer.ImageLayer(source=f"zarr://{group_url}")
    try:
        attrs = requests.get(f"{group_url}/.zattrs", timeout=10).json()
        ms = attrs.get("multiscales", [{}])[0]
        ds_paths = [d["path"] for d in ms.get("datasets", []) if "path" in d]
    except Exception as e:
        print(f"[patch] could not read /.zattrs for {group_url}: {e}; falling back to plain zarr URL")
        return neuroglancer.ImageLayer(source=f"zarr://{group_url}")
    if not ds_paths:
        image = ImageDataInterface(group_url, normalize=normalize, concurrency_limit=16)
        return neuroglancer.ImageLayer(
            source=neuroglancer.LocalVolume(
                data=image.ts,
                dimensions=neuroglancer.CoordinateSpace(
                    names=image.axes_names, units="nm", scales=image.voxel_size,
                ),
                voxel_offset=image.offset,
            ),
        )
    layers = []
    for ds_path in ds_paths:
        try:
            image = ImageDataInterface(f"{group_url}/{ds_path}", normalize=normalize, concurrency_limit=16)
        except Exception as e:
            print(f"[patch] skipping scale {ds_path}: {e}")
            continue
        layers.append(
            neuroglancer.LocalVolume(
                data=image.ts,
                dimensions=neuroglancer.CoordinateSpace(
                    names=image.axes_names, units="nm", scales=image.voxel_size,
                ),
                voxel_offset=image.offset,
            )
        )
    if not layers:
        return neuroglancer.ImageLayer(source=f"zarr://{group_url}")
    print(f"[patch] built ScalePyramid with {len(layers)} levels from {group_url}")
    return neuroglancer.ImageLayer(
        dict(type=neuroglancer.LocalVolume, source=ScalePyramid(layers))
    )

_sp.get_raw_layer = _remote_safe_get_raw_layer
_pipe.get_raw_layer = _remote_safe_get_raw_layer
_ngu.get_raw_layer = _remote_safe_get_raw_layer
print("[patch] get_raw_layer remote-URL-safe (prod-shape via ImageDataInterface)")

# Public URL = Colab proxyPort for the one externally-visible port (nginx).
from google.colab.output import eval_js
PUBLIC_URL = eval_js(f"google.colab.kernel.proxyPort({PUBLIC_PORT})").rstrip("/")
print(f"PUBLIC_URL = {PUBLIC_URL}")

# Override NG-Python's URL emission. By default str(viewer) returns
# http://0.0.0.0:NG_PORT/v/TOKEN/, which the browser can't reach.
# We want the dashboard iframe to load NG from PUBLIC_URL/ng/v/TOKEN/
# (same origin as the dashboard itself, so the Colab auth cookie
# flows in). The path is preserved; only the origin is rewritten.
_NG_BASE = f"{PUBLIC_URL}/ng"
_ng_server_mod._get_server_url = lambda bind_address, port: _NG_BASE
print(f"[patch] NG _get_server_url -> {_NG_BASE}")

# Bind NG to an internal port; nginx forwards /ng/* to it.
neuroglancer.set_server_bind_address("0.0.0.0", bind_port=NG_PORT)

# Skip the dashboard's first-run 'Server Configuration' dialog. The
# cached values (queue, charge group, etc.) are irrelevant in Colab
# since we never submit LSF jobs, but the dashboard gates the UI on
# _server_config_cached. Defaults from globals.py are fine.
g.save_server_config()

g.dataset_path = DATASET
viewer = neuroglancer.Viewer()
g.viewer = viewer
state.NEUROGLANCER_URL = str(viewer)
print(f"NG iframe URL: {state.NEUROGLANCER_URL}")

# Pre-populate g.jobs so the dashboard's /api/process knows about the
# (already running) inference server. host = PUBLIC_URL/infer so the
# NG iframe (same origin) reaches inference via nginx.
class FakeJob:
    def __init__(self, model_name, host):
        self.model_name = model_name
        self.host = host
g.jobs.append(FakeJob(model_name=MODEL_NAME, host=f"{PUBLIC_URL}/infer"))

# Seed the viewer with raw + inference layers so the iframe shows
# something on first load (before any /api/process click).
with viewer.txn() as s:
    s.dimensions = neuroglancer.CoordinateSpace(
        names=["z", "y", "x"], units="nm", scales=[8, 8, 8],
    )
    s.layers["data"] = _remote_safe_get_raw_layer(DATASET)
    s.layers[MODEL_NAME] = neuroglancer.ImageLayer(
        source=f"zarr://{PUBLIC_URL}/infer/{MODEL_NAME}/",
    )

# Dashboard Flask in a background thread on its internal port.
dash_thread = threading.Thread(
    target=lambda: app.run(host="0.0.0.0", port=DASHBOARD_PORT,
                            threaded=True, use_reloader=False, debug=False),
    daemon=True,
)
dash_thread.start()
import time as _t; _t.sleep(2)
print(f"dashboard listening on :{DASHBOARD_PORT} (proxied by nginx at PUBLIC_URL/)")

## 6. Open the dashboard

In [ ]:
print()
print("=" * 70)
print("DASHBOARD URL (this is the only URL you need — click it to open):")
print(f"  {PUBLIC_URL}")
print()
print("Behind the scenes nginx routes /  → dashboard, /ng/* → NG,")
print("/infer/* → inference, all on the same Colab proxyPort URL.")
print("=" * 70)

## 7. Keep-alive

Stop with ▢ to tear down. Drains server logs as they come in.

In [ ]:
import time, select

def drain(proc, label):
    while True:
        r, _, _ = select.select([proc.stdout], [], [], 0)
        if not r: return
        line = proc.stdout.readline()
        if not line: return
        print(f"[{label}] {line}", end="")

try:
    while True:
        drain(server, "server")
        if server.poll() is not None:
            drain(server, "server")
            print(f"\n[server] exited rc={server.returncode}.")
            break
        time.sleep(2)
finally:
    try: server.terminate()
    except Exception: pass
